# 🚗 KUSRC Traffic — YOLO Fine-tuning บน Google Colab
### คู่มือฉบับมือใหม่

**วิธีใช้:**
1. `Runtime` → `Change runtime type` → เลือก **T4 GPU** → Save
2. แก้ `DRIVE_DATASET_PATH` ในเซลล์ถัดไปให้ตรงกับชื่อโฟลเดอร์ใน Google Drive ของคุณ
3. กด `Runtime` → `Run all` (Ctrl+F9)
4. ตอบ Allow เมื่อขอ permission Google Drive
5. รอจนเสร็จ (~30-40 นาที) แล้ว download best.pt

In [ ]:
# ============================================================
# ⚙️  แก้แค่ตรงนี้ตรงเดียวเท่านั้น!
# ============================================================

# Path ในใน Google Drive ที่คุณอัปโหลด dataset ไว้
# ตัวอย่าง: ถ้าสร้างโฟลเดอร์ชื่อ traffic_dataset แล้วใส่โฟลเดอร์ vehicles-coco-2 ไว้ข้างใน
# DRIVE_DATASET_PATH = 'MyDrive/traffic_dataset/vehicles-coco-2'
DRIVE_DATASET_PATH = 'MyDrive/traffic_dataset/vehicles-coco-2'   # << แก้ตรงนี้

# โมเดลที่ใช้เป็น base (n=เร็วมาก, s=สมดุล, m=แม่น)
BASE_MODEL = 'yolo11s.pt'

# การตั้งค่าการเทรน
EPOCHS   = 50   # จำนวนรอบ (เพิ่มถ้าต้องการแม่นขึ้น)
BATCH    = 16   # จำนวนภาพต่อรอบ (ลดเป็น 8 ถ้า RAM เต็ม)
IMGSZ    = 640  # ขนาดภาพ
PATIENCE = 10   # หยุดก่อนถ้า accuracy ไม่ดีขึ้น N รอบ

In [ ]:
# ============================================================
# Step 1: ติดตั้ง ultralytics
# ============================================================
print('กำลังติดตั้ง ultralytics...')
!pip install ultralytics --quiet

import ultralytics
ultralytics.checks()
print('✅ ติดตั้งเสร็จ!')

In [ ]:
# ============================================================
# Step 2: เชื่อมต่อ Google Drive
#         (จะมี popup ขอ permission — กด Allow)
# ============================================================
from google.colab import drive
import os

drive.mount('/content/drive')

dataset_root = f'/content/drive/{DRIVE_DATASET_PATH}'
yaml_path    = f'{dataset_root}/data.yaml'

print(f'Dataset path : {dataset_root}')
print(f'data.yaml    : {yaml_path}')

# ตรวจสอบว่าเจอไฟล์หรือไม่
if not os.path.exists(dataset_root):
    print('❌ ไม่พบโฟลเดอร์ dataset!')
    print('   ตรวจสอบ DRIVE_DATASET_PATH ว่าชื่อตรงกับโฟลเดอร์ใน Drive หรือไม่')
elif not os.path.exists(yaml_path):
    print('❌ ไม่พบ data.yaml!')
    print('   ไฟล์ที่มีในโฟลเดอร์:')
    for f in os.listdir(dataset_root):
        print(f'      {f}')
else:
    print('✅ พบ dataset และ data.yaml แล้ว!')
    print('\n--- เนื้อหา data.yaml ---')
    with open(yaml_path) as f:
        print(f.read())

In [ ]:
# ============================================================
# Step 3: ตรวจสอบ GPU
# ============================================================
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU พร้อมใช้งาน: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('⚠️  ไม่พบ GPU!')
    print('   ไปที่ Runtime → Change runtime type → เลือก T4 GPU')

In [ ]:
# ============================================================
# Step 4: เทรนโมเดล
#         รอได้เลย ใช้เวลา 20-40 นาที
# ============================================================
from ultralytics import YOLO

print(f'โหลด base model: {BASE_MODEL}')
model = YOLO(BASE_MODEL)

print('\n🚀 เริ่มเทรน...')
print(f'   Dataset : {yaml_path}')
print(f'   Epochs  : {EPOCHS}')
print(f'   Batch   : {BATCH}')
print()

results = model.train(
    data=yaml_path,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    device=0,
    project='/content/runs',
    name='traffic_finetune',
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
)

best_path = '/content/runs/traffic_finetune/weights/best.pt'
print(f'\n✅ เทรนเสร็จ! โมเดลอยู่ที่: {best_path}')

In [ ]:
# ============================================================
# Step 5: ดูผลลัพธ์การเทรน
# ============================================================
from ultralytics import YOLO

best_path = '/content/runs/traffic_finetune/weights/best.pt'
best_model = YOLO(best_path)
metrics = best_model.val(data=yaml_path, device=0)

print('\n========== ผลการเทรน ==========')
print(f'mAP50    : {metrics.box.map50:.4f}   (ยิ่งสูงยิ่งดี เป้าหมาย > 0.5)')
print(f'mAP50-95 : {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.p.mean():.4f}   (ความแม่นยำ)')
print(f'Recall   : {metrics.box.r.mean():.4f}   (การจับได้ครบ)')
print('================================')

In [ ]:
# ============================================================
# Step 6: Download best.pt
#         จะมี popup ขึ้นมาให้กด Save
#         นำไฟล์ที่ได้ไปวางในโฟลเดอร์โปรเจกต์
# ============================================================
from google.colab import files
import os

best_path = '/content/runs/traffic_finetune/weights/best.pt'

if os.path.exists(best_path):
    print('กำลัง download best.pt...')
    files.download(best_path)
    print('\n✅ เสร็จแล้ว!')
    print('\nขั้นตอนต่อไป:')
    print('  1. นำไฟล์ best.pt ไปวางในโฟลเดอร์โปรเจกต์')
    print('     C:\\Users\\LENOVO\\OneDrive - KASETSART UNIVERSITY\\Documents\\Traffic Project\\')
    print('  2. เปิด main.py แล้วแก้บรรทัดที่ 37:')
    print('     จาก: MODEL_PATH = resolve_model_path("yolov11n.pt")')
    print('     เป็น: MODEL_PATH = "best.pt"')
    print('  3. รัน: python main.py')
else:
    print('❌ ไม่พบ best.pt — เทรนยังไม่เสร็จหรือมีข้อผิดพลาด')